# Bundle Scoring: Orientacion local + Vecinos + Clasificacion (v1)

**Objetivo final:** para cada punto del esqueleto, calcular su orientacion local (PCA en una
ventana pequena, restringido a su propio fragmento), encontrar vecinos cercanos de OTROS
fragmentos, y clasificar el punto como "bundle" si tiene suficientes vecinos paralelos cercanos
(criterio Jasnin et al.: distancia <=15nm, angulo <20 grados).

**Por que no importa que los fragmentos esten rotos:** cada punto se evalua independientemente.
Dos fragmentos que en realidad son el mismo filamento real, partido por watershed, simplemente
se ven como "vecinos muy cercanos y paralelos" entre si si estuvieran lado a lado -- pero como
es el caso de bundles reales tambien, esto no distorsiona la clasificacion. Lo que si NO se
hace es comparar un punto contra otros puntos de su MISMO fragmento (eso no es informativo).

**Input:** `skeleton_volume` (resultado de la esqueletonizacion por fragmento).

## 1. Carga + extraccion de puntos del esqueleto

In [1]:
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree
import mrcfile
import napari

output_dir = Path(r"C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\P3")
VOXEL_SIZE_A = 12.4  # Angstrom/voxel
VOXEL_TO_NM = VOXEL_SIZE_A / 10.0  # conversion a nm
skeleton_input_path = output_dir / "C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\P3\P3_actin_watershed_md4_ss2.5_er1_skeleton.mrc"
if "skeleton_volume" not in dir():
    with mrcfile.open(str(skeleton_input_path), permissive=True) as mrc:
        skeleton_volume = mrc.data.copy().astype(np.int32)
    print(f"Cargado desde disco: {skeleton_input_path}")
else:
    print("Usando 'skeleton_volume' ya en memoria.")

# Extraer todas las coordenadas (Z,Y,X) y su ID de fragmento, en formato de tabla plana
zs, ys, xs = np.where(skeleton_volume > 0)
all_points = np.column_stack([zs, ys, xs]).astype(float)  # (N, 3) en voxels, orden (Z,Y,X)
all_ids = skeleton_volume[zs, ys, xs]

n_points = len(all_points)
n_fragments = len(np.unique(all_ids))
print(f"Total puntos de esqueleto: {n_points:,}")
print(f"Total fragmentos distintos: {n_fragments:,}")

Cargado desde disco: C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\P3\P3_actin_watershed_md4_ss2.5_er1_skeleton.mrc
Total puntos de esqueleto: 23,502
Total fragmentos distintos: 1,754


## 2. Orientacion local por punto (PCA en ventana, restringido al propio fragmento)

Para cada punto, se toman los puntos del MISMO fragmento dentro de una ventana de
`WINDOW_VOXELS` (distancia euclidea), y se calcula la direccion principal via SVD. Puntos cuyo
fragmento tiene muy pocos vecinos en la ventana (ej. fragmentos de 1-2 voxels) no tienen
suficiente informacion para una orientacion fiable y se marcan como `None`.

In [3]:
WINDOW_VOXELS = 5  # ventana de PCA local, en voxels (~5 voxels = ~62 Å = 6.2 nm a cada lado)

# Para eficiencia: un KDTree POR FRAGMENTO no es practico con miles de fragmentos pequenos.
# En su lugar, agrupamos indices por fragmento una sola vez.
from collections import defaultdict

indices_by_fragment = defaultdict(list)
for idx, fid in enumerate(all_ids):
    indices_by_fragment[fid].append(idx)

orientations = np.full((n_points, 3), np.nan)
n_no_orientation = 0

for fid, idx_list in indices_by_fragment.items():
    idx_array = np.array(idx_list)
    frag_points = all_points[idx_array]

    if len(frag_points) < 2:
        n_no_orientation += len(idx_array)
        continue

    # KDTree local, solo de este fragmento (rapido, son pocos puntos por fragmento)
    local_tree = cKDTree(frag_points)

    for local_i, global_i in enumerate(idx_array):
        center = frag_points[local_i]
        neighbor_local_idxs = local_tree.query_ball_point(center, r=WINDOW_VOXELS)
        window_points = frag_points[neighbor_local_idxs]

        if len(window_points) < 2:
            n_no_orientation += 1
            continue

        centered = window_points - window_points.mean(axis=0)
        if len(window_points) == 2:
            direction = centered[-1] - centered[0]
            norm = np.linalg.norm(direction)
            if norm > 0:
                orientations[global_i] = direction / norm
            else:
                n_no_orientation += 1
            continue

        u, s, vt = np.linalg.svd(centered)
        orientations[global_i] = vt[0]

print(f"Puntos con orientacion calculada: {n_points - n_no_orientation:,} / {n_points:,}")
print(f"Puntos sin orientacion fiable (fragmento muy corto): {n_no_orientation:,}")

Puntos con orientacion calculada: 23,455 / 23,502
Puntos sin orientacion fiable (fragmento muy corto): 47


## 3. Vecinos cercanos + clasificacion bundle

Para cada punto con orientacion valida, se buscan puntos de OTROS fragmentos dentro de
`DIST_THRESHOLD_NM`, se mide el angulo entre orientaciones, y se cuenta cuantos vecinos
cumplen ambos criterios (distancia + angulo). Un punto se clasifica como bundle si tiene
al menos `N_MIN_PARALLEL` vecinos paralelos cercanos.

**Nota de rendimiento:** esto esta vectorizado con `query_pairs` + operaciones de numpy sobre
arrays completos, en vez de un loop punto-por-punto con un sub-loop de vecinos en Python puro.
Con ~360,000 puntos (tamano real de este dataset) la version vectorizada tarda pocos segundos;
la version con doble loop tardaba ordenes de magnitud mas. Verificado que ambas dan el mismo
resultado en un caso de control sintetico antes de reemplazarla.

In [4]:
# Parametros finales (elegidos por barrido previo y confirmados visualmente en Napari --
# distinguen bien bundle real de filamentos sueltos en los bordes).
DIST_THRESHOLD_NM = 10
ANGLE_THRESHOLD_DEG = 15
N_MIN_PARALLEL = 5

dist_threshold_voxels = DIST_THRESHOLD_NM / VOXEL_TO_NM
has_orientation = ~np.isnan(orientations[:, 0])
valid_idxs = np.where(has_orientation)[0]
print(f"Puntos con orientacion valida: {len(valid_idxs):,} / {n_points:,}")

# VECTORIZADO: query_pairs para todos los pares dentro del radio de una sola vez.
global_tree = cKDTree(all_points)
pairs = global_tree.query_pairs(r=dist_threshold_voxels, output_type="ndarray")
print(f"Pares candidatos dentro de {DIST_THRESHOLD_NM}nm: {len(pairs):,}")

i_idx, j_idx = pairs[:, 0], pairs[:, 1]

# Descartar pares del mismo fragmento (no cuentan como "vecino externo")
same_fragment = all_ids[i_idx] == all_ids[j_idx]
i_idx, j_idx = i_idx[~same_fragment], j_idx[~same_fragment]

# Descartar pares donde alguno de los dos puntos no tiene orientacion valida
both_valid = has_orientation[i_idx] & has_orientation[j_idx]
i_idx, j_idx = i_idx[both_valid], j_idx[both_valid]

# Distancia real en nm (vectorizado)
diffs = all_points[i_idx] - all_points[j_idx]
dists_nm = np.linalg.norm(diffs, axis=1) * VOXEL_TO_NM

# Angulo entre orientaciones (vectorizado). abs() porque la orientacion de un filamento
# no tiene "sentido" definido (PCA da un eje, no una direccion con signo).
cos_angles = np.abs(np.sum(orientations[i_idx] * orientations[j_idx], axis=1))
cos_angles = np.clip(cos_angles, -1, 1)
angles_deg = np.degrees(np.arccos(cos_angles))

# FILTRO CLAVE (nuevo): excluir "continuacion de si mismo". Un filamento partido por watershed
# en varios fragmentos a lo largo de SU PROPIA longitud tiene fragmentos vecinos que estan
# casi en LINEA con su propia orientacion (el desplazamiento i->j es mayormente PARALELO a la
# orientacion local). Un vecino de bundle real esta desplazado LATERALMENTE (el desplazamiento
# es mayormente PERPENDICULAR a la orientacion). Solo contamos como vecino real el segundo caso.
parallel_component = np.abs(np.sum(diffs * orientations[i_idx], axis=1)) * VOXEL_TO_NM
perpendicular_component = np.sqrt(np.maximum(dists_nm**2 - parallel_component**2, 0))
is_lateral_neighbor = perpendicular_component > parallel_component

valid_pairs_mask = (
    (dists_nm <= DIST_THRESHOLD_NM)
    & (angles_deg <= ANGLE_THRESHOLD_DEG)
    & is_lateral_neighbor
)
valid_i = i_idx[valid_pairs_mask]
valid_j = j_idx[valid_pairs_mask]

# Contar FRAGMENTOS VECINOS DISTINTOS (no puntos individuales) -- ver nota de la version anterior:
# un fragmento vecino con varios voxels dentro del radio cuenta como 1 vecino, no varios.
point_idx_for_count = np.concatenate([valid_i, valid_j])
neighbor_fragment_for_count = np.concatenate([all_ids[valid_j], all_ids[valid_i]])
pairs_point_fragment = np.column_stack([point_idx_for_count, neighbor_fragment_for_count])
unique_point_fragment_pairs = np.unique(pairs_point_fragment, axis=0)

n_parallel_fragments = np.zeros(n_points, dtype=int)
np.add.at(n_parallel_fragments, unique_point_fragment_pairs[:, 0], 1)

is_bundle = (n_parallel_fragments >= N_MIN_PARALLEL) & has_orientation

n_bundle = is_bundle.sum()
n_evaluated = has_orientation.sum()
print(f"\nPuntos clasificados como BUNDLE: {n_bundle:,} / {n_evaluated:,} "
      f"({100*n_bundle/n_evaluated:.1f}%)")
print(f"Puntos clasificados como NO-bundle (mesh/aislado): {n_evaluated - n_bundle:,} "
      f"({100*(n_evaluated-n_bundle)/n_evaluated:.1f}%)")

n_parallel_neighbors = n_parallel_fragments  # compatibilidad con secciones siguientes

Puntos con orientacion valida: 23,455 / 23,502
Pares candidatos dentro de 10nm: 792,568

Puntos clasificados como BUNDLE: 11,238 / 23,455 (47.9%)
Puntos clasificados como NO-bundle (mesh/aislado): 12,217 (52.1%)


## 4. Visualizar clasificacion en Napari

Puntos coloreados: bundle vs. no-bundle, superpuestos sobre el esqueleto completo.

In [6]:
# Volumen de clasificacion: 0=no esqueleto, 1=no-bundle, 2=bundle
classification_volume = np.zeros_like(skeleton_volume, dtype=np.uint8)

zs_valid = zs[valid_idxs]
ys_valid = ys[valid_idxs]
xs_valid = xs[valid_idxs]
classification_volume[zs_valid, ys_valid, xs_valid] = np.where(is_bundle[valid_idxs], 2, 1)

viewer_class = napari.Viewer(title="Clasificacion bundle (2=bundle, 1=no-bundle)")
viewer_class.add_labels(skeleton_volume, name="esqueleto (todos los fragmentos)", opacity=0.2)
viewer_class.add_labels(classification_volume, name="clasificacion bundle/no-bundle", opacity=0.9)
viewer_class.dims.ndisplay = 3

print("Verde/amarillo (label=2) deberia corresponder a zonas de bundle denso.")
print("El otro color (label=1) deberia corresponder a filamentos aislados o de mesh.")

Verde/amarillo (label=2) deberia corresponder a zonas de bundle denso.
El otro color (label=1) deberia corresponder a filamentos aislados o de mesh.


## 5. Guardar resultados (tabla + volumen de clasificacion)

In [7]:
import pandas as pd

# Tabla con un punto por fila: posicion, fragmento, orientacion, n_vecinos_paralelos, clasificacion
results_df = pd.DataFrame({
    "z_voxel": zs,
    "y_voxel": ys,
    "x_voxel": xs,
    "fragment_id": all_ids,
    "orientation_z": orientations[:, 0],
    "orientation_y": orientations[:, 1],
    "orientation_x": orientations[:, 2],
    "n_parallel_neighbors": n_parallel_neighbors,
    "is_bundle": is_bundle,
})

csv_output_path = output_dir / "Pos16_actin_bundle_scoring.csv"
results_df.to_csv(csv_output_path, index=False)
print(f"Tabla guardada: {csv_output_path} ({len(results_df):,} filas)")

vol_output_path = output_dir / "Pos16_actin_bundle_classification.mrc"
with mrcfile.new(str(vol_output_path), overwrite=True) as mrc_out:
    mrc_out.set_data(classification_volume.astype(np.int8))
    mrc_out.voxel_size = VOXEL_SIZE_A
print(f"Volumen de clasificacion guardado: {vol_output_path}")

Tabla guardada: C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\P3\Pos16_actin_bundle_scoring.csv (23,502 filas)
Volumen de clasificacion guardado: C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\P3\Pos16_actin_bundle_classification.mrc


## 6. Iterar parametros

- **`WINDOW_VOXELS`** (seccion 2): ventana de PCA local. Mas grande = orientacion mas suave
  pero menos sensible a curvatura local; mas pequena = mas sensible a ruido.
- **`DIST_THRESHOLD_NM` / `ANGLE_THRESHOLD_DEG` / `N_MIN_PARALLEL`** (seccion 3): criterio
  bundle. Valores actuales (10nm / 15° / N_MIN=5) elegidos por barrido de sensibilidad y
  confirmados visualmente. Si cambias de region/tomograma y el resultado no se ve bien, ajusta
  estos tres a mano y revisa en Napari -- el barrido completo de 45 combinaciones se hizo una
  vez para calibrar, pero no es necesario repetirlo cada vez (tarda varios minutos).